# AMMS 302 | Week 3: Finding Insights with pandas
**รายวิชา ปฏิบัติการสารสนเทศด้านสุขภาพ (Workshop in Health Informatics)**
**หลักสูตรวิทยาศาสตรบัณฑิต สาขาวิชาวิทยาศาสตร์การแพทย์ คณะสหเวชศาสตร์**

---

### 🎯 วัตถุประสงค์การเรียนรู้ (CLO2)
1. นักศึกษาสามารถนำเข้าและตรวจสอบคุณภาพข้อมูลสุขภาพเบื้องต้นได้
2. นักศึกษาสามารถใช้ Boolean Indexing ในการคัดกรองประชากรผู้ป่วยเป้าหมาย (Cohort Selection) ตามเงื่อนไขทางคลินิกและระบาดวิทยาได้
3. นักศึกษาสามารถประมวลผลข้อมูลสถิติสุขภาพด้วยคำสั่ง Groupby และ Aggregation ได้
4. นักศึกษาสามารถสืบค้นและทำความสะอาดข้อมูลที่ขัดแย้งทางตรรกะคลินิก (Logical Data Validation) ได้
> **Self-study บน Windows:** ติดตั้งด้วย scoop + uv ดูเซลล์ Setup ด้านบน — PySynthea แทน Synthea Java: [TIET-AI/tietai-synthea](https://github.com/TIET-AI/tietai-synthea) | [PyPI](https://pypi.org/project/tietai-synthea/)


## 🛠️ Setup สำหรับ Windows ด้วย scoop + uv (Self-study, รันออฟไลน์ได้)

> **ทำตามได้บน Windows 10/11 โดยไม่ต้องพึ่งเซิร์ฟเวอร์มหาวิทยาลัย** — ใช้ PowerShell (ไม่ใช่ CMD)

**1. ติดตั้ง scoop (ครั้งเดียว):**
```powershell
Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser
Invoke-RestMethod -Uri https://get.scoop.sh | Invoke-Expression
scoop --version   # ทดสอบว่าติดตั้งสำเร็จ
```
คู่มือทางการ: [scoop.sh](https://scoop.sh) | [Scoop docs (GitHub)](https://github.com/ScoopInstaller/Scoop)

**2. ติดตั้ง git + uv ผ่าน scoop:**
```powershell
scoop install git uv
uv --version
```
เอกสารทางการ: [uv installation](https://docs.astral.sh/uv/getting-started/installation/) | [uv — managing projects](https://docs.astral.sh/uv/guides/projects/)

**3. สร้างโปรเจกต์และติดตั้งไลบรารี (ในโฟลเดอร์ health-informatics ของคุณ):**
```powershell
D:
mkdir health-informatics; cd health-informatics
uv init
uv add jupyterlab pandas faker tietai-synthea
# หรือถ้ามีโปรเจกต์อยู่แล้ว: uv add tietai-synthea
```

**4. เปิด JupyterLab แบบ reproducible:**
```powershell
uv run jupyter lab
# เปิดเบราว์เซอร์ที่ http://localhost:8888 แล้วเปิดไฟล์ .ipynb นี้
```

**หมายเหตุ PySynthea (แทน Synthea Java เดิม):**
- PyPI: [tietai-synthea](https://pypi.org/project/tietai-synthea/) — import ชื่อ `synthea`, คำสั่ง CLI `synthea`
- GitHub: [TIET-AI/tietai-synthea](https://github.com/TIET-AI/tietai-synthea) (Quick Start, API, disease modules)
- Paper: [PySynthea (PDF)](https://tiet.ai/pdf/pysynthea-paper.pdf) | Blog: [tiet.ai/blog/py-synthea](https://tiet.ai/blog/py-synthea/)
- ไม่ต้องติดตั้ง Java/JVM — รันด้วย `uv run synthea -p 10` หรือ Python API `from synthea import Generator, GeneratorOptions`
- ข้อมูลสังเคราะห์จะส่งออกเป็น FHIR R4 JSON Bundle — สเปกทางการ: [HL7 FHIR R4](https://www.hl7.org/fhir/) | [FHIR Patient](https://www.hl7.org/fhir/patient.html) | [FHIR Bundle](https://www.hl7.org/fhir/bundle.html)

**อ้างอิง pandas / Jupyter ที่ใช้ในแล็บนี้:**
- [pandas docs](https://pandas.pydata.org/docs/) | [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) | [pandas I/O](https://pandas.pydata.org/pandas-docs/stable/reference/io.html) | [Indexing & selecting](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [JupyterLab docs](https://jupyterlab.readthedocs.io/en/latest/)

> ทุกเซลล์ในโน้ตบุ๊กนี้รันด้วย `uv run` จึงล็อกเวอร์ชันผ่าน `uv.lock` — ส่งโปรเจกต์ให้อาจารย์แล้วรันซ้ำได้ผลเดิม (reproducible)


---
## 1. การนำเข้าและเตรียมความพร้อมข้อมูล (Data Ingestion)
เริ่มจากการนำเข้าไลบรารี `pandas` และโหลดไฟล์ข้อมูลที่ใช้ในสัปดาห์นี้ ได้แก่:
- `prescriptions.csv` (ข้อมูลใบสั่งยาทางคลินิก)
- `practices_registry.csv` (ทะเบียนรายชื่อคลินิก)

In [ ]:
import pandas as pd
import numpy as np

# นำเข้าข้อมูล
df_prescriptions = pd.read_csv('prescriptions.csv')
df_practices = pd.read_csv('practices_registry.csv')

print("Prescriptions data shape:", df_prescriptions.shape)
print("Practices registry shape:", df_practices.shape)

### 🔎 ตรวจสอบโครงสร้างข้อมูลเบื้องต้นด้วย `.head()`, `.info()` และ `.describe()`

In [ ]:
# แสดงข้อมูล 5 แถวแรก
df_prescriptions.head()

In [ ]:
# ตรวจสอบชนิดตัวแปรและข้อมูลสูญหาย (Missing Values)
df_prescriptions.info()

In [ ]:
# สรุปข้อมูลทางสถิติเบื้องต้น
df_prescriptions.describe()

---
## 2. การกรองข้อมูลแบบมีเงื่อนไข (Boolean Indexing & Cohort Selection)

การกรองผู้ป่วยตามเกณฑ์ระบาดวิทยา เช่น การเลือกผู้ป่วยที่ได้ยารักษาเบาหวาน `METFORMIN` หรือการคัดกรองกลุ่มเสี่ยงตามช่วงอายุและเพศ

⚠️ **กฎข้อบังคับของตรรกศาสตร์ pandas:**
- ต้องใช้ตัวดำเนินการแบบบิต: `&` (AND), `|` (OR), `~` (NOT)
- **ต้องใส่เครื่องหมายวงเล็บครอบแต่ละเงื่อนไขย่อยเสมอ**

### ตัวอย่างที่ 2.1: กรองข้อมูลเฉพาะผู้ป่วยหญิงที่มีอายุตั้งแต่ 60 ปีขึ้นไป และมีค่าน้ำตาลสะสม (HbA1c) สูงกว่า 7.0%

In [ ]:
condition = (df_prescriptions['gender'] == 'female') & \
            (df_prescriptions['age'] >= 60) & \
            (df_prescriptions['hba1c'] > 7.0)

df_high_risk = df_prescriptions.loc[condition]
print(f"จำนวนผู้ป่วยหญิงสูงวัยกลุ่มเสี่ยงสูง: {len(df_high_risk)} ราย")
df_high_risk.head()

### ตัวอย่างที่ 2.2: กรองข้อมูลผู้ป่วยที่ได้ยารักษาโรคเบาหวานเป้าหมาย (`METFORMIN`, `INSULIN`, หรือ `GLIPIZIDE`) โดยใช้คำสั่ง `.isin()`

In [ ]:
diabetic_drugs = ['METFORMIN', 'INSULIN', 'GLIPIZIDE']
df_diabetic = df_prescriptions[df_prescriptions['DRUG'].isin(diabetic_drugs)]
print(f"จำนวนใบสั่งยารักษาเบาหวาน: {len(df_diabetic)} รายการ")
df_diabetic['DRUG'].value_counts()

---
## 3. การจัดกลุ่มและการประมวลผลข้อมูล (Data Aggregation & Groupby)
กระบวนการ **Split-Apply-Combine** เพื่อหาข้อสรุปเชิงสถิติตามกลุ่มที่สนใจ เช่น คลินิกหรือรหัสโรคสากล

### ตัวอย่างที่ 3.1: หาค่าน้ำตาลสะสม (HbA1c) เฉลี่ย, สูงสุด และจำนวนประวัติการตรวจแยกตามรหัสคลินิก (`practice_code`)

In [ ]:
clinic_summary = df_prescriptions.groupby('practice_code')['hba1c'].agg(['mean', 'max', 'count'])
clinic_summary

### ตัวอย่างที่ 3.2: ตรวจหาจำนวนประชากรผู้ป่วยที่ไม่มีประวัติซ้ำกัน (Unique Patients) ในแต่ละพื้นที่คลินิก

In [ ]:
patient_counts = df_prescriptions.groupby('practice_code')['patient_id'].nunique()
print(patient_counts)

---
## 4. การตรวจสอบความสอดคล้องเชิงตรรกะคลินิก (Logical Data Validation)
หนึ่งในงานที่ต้องทำในขั้นเตรียมข้อมูล (Data Pre-processing) คือ การหาจุดผิดพลาดเชิงตรรกะเวลาในประวัติผู้ป่วย เช่น:
- วันจำหน่าย (`dischtime`) เกิดขึ้นก่อนวันรับเข้าพยาบาล (`admittime`)
- ระยะเวลานอนโรงพยาบาลติดลบ (`los_days` < 0)

In [ ]:
# 1. แปลงวันที่จากสตริงเป็นวัตถุเวลา Datetime
df_prescriptions['admittime'] = pd.to_datetime(df_prescriptions['admittime'])
df_prescriptions['dischtime'] = pd.to_datetime(df_prescriptions['dischtime'])

# 2. สืบค้นหาแถวที่มีตรรกะวันเวลาขัดแย้งกัน
invalid_dates = df_prescriptions[df_prescriptions['dischtime'] < df_prescriptions['admittime']]
print(f"พบข้อมูลวันที่ผิดพลาดเชิงตรรกะจำนวน: {len(invalid_dates)} แถว")
invalid_dates[['patient_id', 'admittime', 'dischtime', 'los_days']]

---
## 📝 แบบฝึกหัดปฏิบัติด้วยตนเอง (Jupyter Exercise)

ให้นักศึกษาตอบคำถามและเขียนโค้ดแสดงผลลัพธ์ของโจทย์ต่อไปนี้ส่งผู้สอนในระบบประเมิน:

### **โจทย์ข้อที่ 1:**
จงเขียนโค้ดเพื่อค้นหาว่าในชุดข้อมูลใบสั่งยานี้ มียารักษาโรคชนิดใดบ้างที่ถูกจ่ายให้กับผู้ป่วยโรคหัวใจเต้นผิดจังหวะ (รหัสวินิจฉัย `icd_code` เท่ากับ **`I48`**) พร้อมทั้งนับจำนวนความถี่ของการจ่ายยาแต่ละตัวชนิดนั้น


In [ ]:
# เขียนคำตอบข้อที่ 1 ของนักศึกษาลงในส่วนนี้
df_i48 = ...


### **โจทย์ข้อที่ 2:**
คอลัมน์ `hba1c` มีข้อมูลสูญหาย (Null Values) อยู่บางส่วน จงเขียนโค้ดเพื่อคำนวณหาค่าเฉลี่ยของ HbA1c ของประชากรทั้งหมด และทำการแทนที่ข้อมูลช่องว่าง (Missing data imputation) ในคอลัมน์นั้นด้วยค่าเฉลี่ยที่คำนวณได้ จากนั้นให้พิมพ์ผลรวมของข้อมูลที่เป็นค่าว่างหลังทำการแทนที่เสร็จสิ้นลงบนหน้าจอเพื่อทวนสอบความถูกต้อง

In [ ]:
# เขียนคำตอบข้อที่ 2 ของนักศึกษาลงในส่วนนี้
hba1c_mean = ...
